# Prepare a small particle dataset

Crop matching tomogram and instance-label arrays, map intensities to uint8,
and save a Zarr pair for particle extraction. Supply your own source data;
this example does not download a dataset.

Start Jupyter from the repository root with
`uv run --extra notebooks jupyter lab`. Edit the configuration before running.
The final step writes particle files using the project's extraction CLI.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src").is_dir()
)
os.chdir(PROJECT_ROOT)


In [ ]:
import subprocess
import sys

import dask.array as da
import matplotlib.pyplot as plt
import numpy as np
from pptools.io import ZarrReader, ZarrWriter, load_nc_arr


## Configure inputs and crop

Inputs can be Zarr stores or NetCDF file patterns. Both arrays must use the
same `(z, y, x)` axes; the mask must contain integer instance IDs, with zero
for background. Choose a crop within the source dimensions.

The clipping bounds below follow the original int16 dataset convention;
adjust them for your own scanner and intensity range. For an existing uint8
tomogram, set `INTENSITY_RANGE = None` to preserve its values.

In [ ]:
TOMO_PATH = "data/GOH_6_250/tomo.zarr"
MASK_PATH = "data/GOH_6_250/mask.zarr"
CROP = (slice(0, 256), slice(0, 256), slice(0, 256))
INTENSITY_RANGE = (10000, 32767)
OUTPUT_DIR = Path("data/toy_data_GOH_6_250")
PARTICLE_DIR = OUTPUT_DIR / "sim_data_uint8"


In [ ]:
def read_array(path):
    if str(path).rstrip("/").endswith(".zarr"):
        return ZarrReader(str(path)).get_dask_array()
    return load_nc_arr(str(path))

source_tomo = read_array(TOMO_PATH)
source_mask = read_array(MASK_PATH)
assert source_tomo.ndim == 3 and source_tomo.shape == source_mask.shape
assert np.issubdtype(source_mask.dtype, np.integer)

tomo = source_tomo[CROP]
mask = source_mask[CROP]
assert all(size > 0 for size in tomo.shape), "Choose a crop within the input volume."
if INTENSITY_RANGE is None:
    assert tomo.dtype == np.uint8, "Set clipping bounds for non-uint8 input."
else:
    lower, upper = INTENSITY_RANGE
    assert upper > lower
    tomo = da.clip((tomo.astype(np.float32) - lower) * 255 / (upper - lower), 0, 255).astype(np.uint8)

print(f"Crop: {tomo.shape}; tomogram: {tomo.dtype}; mask: {mask.dtype}")


## Preview the crop

In [ ]:
z = tomo.shape[0] // 2
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(tomo[z].compute(), cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Tomogram, z={z}")
label_slice = mask[z].compute()
axes[1].imshow(np.ma.masked_where(label_slice == 0, label_slice % 20), cmap="tab20", vmin=0, vmax=19, interpolation="nearest")
axes[1].set_title("Instance labels (colors repeat)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Save the prepared arrays

This writes `tomo_subset.zarr` and `mask_subset.zarr` under `OUTPUT_DIR`.
Existing stores at these paths are replaced when this cell is rerun.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, array in (("tomo_subset", tomo), ("mask_subset", mask)):
    writer = ZarrWriter(
        path=str(OUTPUT_DIR / f"{name}.zarr"),
        shape=array.shape,
        dtype=array.dtype,
        chunks=tuple(min(128, size) for size in array.shape),
        shards=tuple(min(128, size) for size in array.shape),
        comp_level=1,
        overwrite=True,
    )
    writer.write(array)

saved_tomo = ZarrReader(str(OUTPUT_DIR / "tomo_subset.zarr")).get_dask_array()
saved_mask = ZarrReader(str(OUTPUT_DIR / "mask_subset.zarr")).get_dask_array()
print(f"Saved matching volumes: {saved_tomo.shape}, {saved_mask.shape}")


## Extract particle meshes and volumes

Run extraction in a separate Python process so multiprocessing works from
Jupyter. This cell writes numbered `.obj` and `.npy.gz` files to
`PARTICLE_DIR`; rerunning can replace particles with matching IDs. Objects
intersecting the crop boundary may be incomplete: review extracted particles
with `uv run --extra labeller python labeller.py` and save their labels.

In [ ]:
n_labels = int(saved_mask.max().compute())
subprocess.run(
    [
        sys.executable, "-m", "src.create_simulation_data_zarr",
        "--tomo_path", str(OUTPUT_DIR / "tomo_subset.zarr"),
        "--mask_path", str(OUTPUT_DIR / "mask_subset.zarr"),
        "--output_dir", str(PARTICLE_DIR),
        "--n_labels", str(n_labels),
        "--parallel", "2",
    ],
    cwd=PROJECT_ROOT,
    check=True,
)
